In [2]:

import json
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Set, Tuple


# ---- Configurable PII types (JSONL -> table label) ----
PII_TYPES: List[Tuple[str, str]] = [
    ("EMAIL", "Email Address"),
    ("PHONE", "Phone Number"),
    ("USERNAME", "User Name"),
    ("PERSON_NAME", "Person Name"),
    ("POSTAL_ADDRESS", "Postal Address"),
]


def _db_key_from_record(rec: Mapping) -> str:
    """
    Prefer db_path from JSONL, fall back to 'unknown_db' if missing.
    Example db_path: 'selectedDBs\\A1_msgstore.db' -> 'A1_msgstore'
    """
    db_path = str(rec.get("db_path", "")).strip()
    if not db_path:
        return "unknown_db"
    return Path(db_path).stem


def _has_any_pii(rec: Mapping) -> bool:
    """
    Treat a PII type as present in a DB if the record has at least one entity.
    Uses the PII list when available; falls back to Num_of_PII.
    """
    pii_list = rec.get("PII", None)
    if isinstance(pii_list, list):
        return len(pii_list) > 0
    try:
        return int(rec.get("Num_of_PII", 0)) > 0
    except Exception:
        return False


def collect_db_sets(folder: Path, pii_types: Iterable[str]) -> Dict[str, Set[str]]:
    """
    Returns: pii_type -> {db_key, ...} where that pii_type appears at least once.
    """
    wanted = set(pii_types)
    db_sets: Dict[str, Set[str]] = defaultdict(set)

    files = sorted(folder.glob("*.jsonl"))
    if not files:
        raise FileNotFoundError(f"No .jsonl files found in: {folder}")

    for fp in files:
        with fp.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                pii_type = str(rec.get("PII_type", "")).strip()
                if pii_type not in wanted:
                    continue
                if _has_any_pii(rec):
                    db_sets[pii_type].add(_db_key_from_record(rec))

    for t in wanted:
        db_sets.setdefault(t, set())

    return db_sets


@dataclass(frozen=True)
class CoverageRow:
    label: str
    gt: int
    sys: int
    overlap: int
    coverage_pct: float


def compute_coverage(
    gt_sets: Dict[str, Set[str]],
    sys_sets: Dict[str, Set[str]],
    pii_types: List[Tuple[str, str]],
) -> List[CoverageRow]:
    rows: List[CoverageRow] = []
    for key, label in pii_types:
        dg = gt_sets.get(key, set())
        ds = sys_sets.get(key, set())
        inter = dg & ds
        cov = (len(inter) / len(dg) * 100.0) if len(dg) else 0.0
        rows.append(CoverageRow(label, len(dg), len(ds), len(inter), cov))
    return rows


def render_latex_tabular(rows: List[CoverageRow]) -> str:
    """
    Print only the tabular environment (as requested).
    """
    lines: List[str] = []
    lines.append(r"\begin{tabular}{|l|p{1.2cm}|p{1.5cm}|p{1.0cm}|p{1.2cm}|}")
    lines.append(r"\hline")
    lines.append(
        r"\textbf{PII Type} &"
        r"\textbf{DBs with PII (GT)} &"
        r"\textbf{DBs with discoveries (System)} &"
        r"\textbf{Overlap} &"
        r"\textbf{Coverage} \\"
    )
    lines.append(r"\hline")

    for r in rows:
        lines.append(
            f"{r.label} & {r.gt} & {r.sys} & {r.overlap} & {r.coverage_pct:.1f}\\% \\\\"
        )
        lines.append(r"\hline")

    lines.append(r"\end{tabular}")
    return "\n".join(lines)


def render_plain_text_table(rows: List[CoverageRow]) -> str:
    """
    Simple fixed-width table for quick reading in terminal.
    """
    headers = ["PII Type", "GT DBs", "System DBs", "Overlap", "Coverage"]
    data = [
        [r.label, str(r.gt), str(r.sys), str(r.overlap), f"{r.coverage_pct:.1f}%"]
        for r in rows
    ]

    # compute column widths
    widths = [len(h) for h in headers]
    for row in data:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(cell))

    def fmt_row(row: List[str]) -> str:
        return " | ".join(cell.ljust(widths[i]) for i, cell in enumerate(row))

    sep = "-+-".join("-" * w for w in widths)

    out: List[str] = []
    out.append(fmt_row(headers))
    out.append(sep)
    for row in data:
        out.append(fmt_row(row))
    return "\n".join(out)


def main() -> None:
    # Define these inside main so importing this module has no side effects.
    SYSTEM_DIR = Path(r"..\normalized_PII_results\GPT-5.1\db_level")
    GT_DIR = Path(r"..\normalized_PII_results\ground_truth\db_level")
    
    gt_sets = collect_db_sets(GT_DIR, [k for k, _ in PII_TYPES])
    sys_sets = collect_db_sets(SYSTEM_DIR, [k for k, _ in PII_TYPES])

    rows = compute_coverage(gt_sets, sys_sets, PII_TYPES)

    print("PLAIN TEXT TABLE\n")
    print(render_plain_text_table(rows))
    print("\nLATEX TABULAR\n")
    print(render_latex_tabular(rows))


if __name__ == "__main__":
    main()


PLAIN TEXT TABLE

PII Type       | GT DBs | System DBs | Overlap | Coverage
---------------+--------+------------+---------+---------
Email Address  | 6      | 7          | 6       | 100.0%  
Phone Number   | 9      | 7          | 6       | 66.7%   
User Name      | 6      | 4          | 4       | 66.7%   
Person Name    | 12     | 11         | 9       | 75.0%   
Postal Address | 2      | 1          | 1       | 50.0%   

LATEX TABULAR

\begin{tabular}{|l|p{1.2cm}|p{1.5cm}|p{1.0cm}|p{1.2cm}|}
\hline
\textbf{PII Type} &\textbf{DBs with PII (GT)} &\textbf{DBs with discoveries (System)} &\textbf{Overlap} &\textbf{Coverage} \\
\hline
Email Address & 6 & 7 & 6 & 100.0\% \\
\hline
Phone Number & 9 & 7 & 6 & 66.7\% \\
\hline
User Name & 6 & 4 & 4 & 66.7\% \\
\hline
Person Name & 12 & 11 & 9 & 75.0\% \\
\hline
Postal Address & 2 & 1 & 1 & 50.0\% \\
\hline
\end{tabular}
